# Customer Churn Prediction with Python

**Complete Machine Learning workflow for the YouTube tutorial**

This notebook uses the Telco Customer Churn dataset and demonstrates:
- Feature/target separation
- Train/test split
- Preprocessing categorical and numerical features
- Logistic Regression
- Pipeline
- Evaluation with accuracy, precision, recall and F1-score
- Confusion matrix and ROC-AUC
- Prediction for a brand-new customer
- Probability-based churn prediction
- Threshold tuning

> **Dataset note:** The original IBM repository still contains the dataset, but the old filename `WA_Fn-UseC_-Telco-Customer-Churn.csv` is not the path used by IBM's current archived repository. The code below uses IBM's actual file path: `Telco-Customer-Churn.csv`.


## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    RocCurveDisplay
)

print("Libraries imported successfully.")


## 2. Load the dataset

### Robust approach for Google Colab

We first try the IBM dataset URL. If the URL is unavailable in your environment, the cell automatically lets you upload the CSV file instead.

**Important:** The IBM repository uses the filename `Telco-Customer-Churn.csv`, not `WA_Fn-UseC_-Telco-Customer-Churn.csv`.


In [ ]:
# IBM's archived dataset path
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

try:
    df = pd.read_csv(url)
    print("Dataset loaded from IBM GitHub.")
except Exception as e:
    print("Automatic download failed.")
    print("Error:", e)
    print("\nPlease upload Telco-Customer-Churn.csv from your computer.")

    from google.colab import files
    uploaded = files.upload()
    file_name = next(iter(uploaded))
    df = pd.read_csv(file_name)
    print("Dataset loaded from uploaded file.")

print("Dataset shape:", df.shape)
df.head()


## 3. Inspect the data

In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes)

print("\nMissing values:")
display(df.isnull().sum())

print("\nDataset shape:", df.shape)


## 4. Clean `TotalCharges`

`TotalCharges` can contain blank strings. Convert it to numeric; invalid values become missing values and will be handled later by the preprocessing pipeline.


In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print("Missing values in TotalCharges:", df["TotalCharges"].isnull().sum())


## 5. Separate features and target

In [ ]:
X = df.drop("Churn", axis=1)
y = df["Churn"].map({"No": 0, "Yes": 1})

print("Features shape:", X.shape)
print("Target shape:", y.shape)

display(X.head())
display(y.head())


## 6. Remove Customer ID

In [ ]:
# CustomerID identifies the customer but does not represent useful behavioral information.
X = X.drop("customerID", axis=1)

print("CustomerID removed.")
print("Number of features:", X.shape[1])


## 7. Understand the target

In [ ]:
print("Churn distribution:")
display(y.value_counts())

print("\nChurn percentage:")
display((y.value_counts(normalize=True) * 100).round(2))


## 8. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])


## 9. Identify numerical and categorical features

In [ ]:
numerical_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)


## 10. Build the preprocessing engine

In [ ]:
numerical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_pipeline, numerical_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

print("Preprocessor created.")


## 11. Build the Logistic Regression model

Logistic Regression is a strong, simple baseline for binary classification.


In [ ]:
model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

print("Model created.")


## 12. Create the complete ML Pipeline

In [ ]:
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model)
])

pipeline


## 13. Train the model

In [ ]:
pipeline.fit(X_train, y_train)

print("Model trained successfully.")


## 14. Predict on unseen test data

In [ ]:
y_pred = pipeline.predict(X_test)

print("First 20 predictions:")
print(y_pred[:20])


## 15. Accuracy

In [ ]:
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print(f"Accuracy: {accuracy * 100:.2f}%")


## 16. Classification Report

In [ ]:
print(classification_report(
    y_test,
    y_pred,
    target_names=["Stayed", "Churned"]
))


## 17. Precision, Recall and F1-Score

In [ ]:
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")


## 18. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Stayed", "Churned"]
)

disp.plot()
plt.title("Customer Churn Confusion Matrix")
plt.show()


## 19. ROC-AUC

In [ ]:
y_proba = pipeline.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, y_proba)
print(f"ROC-AUC: {roc_auc:.4f}")

RocCurveDisplay.from_predictions(y_test, y_proba)
plt.title("ROC Curve")
plt.show()


## 20. Predict a brand-new customer

The columns below represent a hypothetical customer.

We will ask:

> **Will this customer churn?**

The model returns both the predicted class and the probability of churn.


In [ ]:
new_customer = pd.DataFrame({
    "gender": ["Female"],
    "SeniorCitizen": [0],
    "Partner": ["No"],
    "Dependents": ["No"],
    "tenure": [5],
    "PhoneService": ["Yes"],
    "MultipleLines": ["No"],
    "InternetService": ["Fiber optic"],
    "OnlineSecurity": ["No"],
    "OnlineBackup": ["No"],
    "DeviceProtection": ["No"],
    "TechSupport": ["No"],
    "StreamingTV": ["No"],
    "StreamingMovies": ["No"],
    "Contract": ["Month-to-month"],
    "PaperlessBilling": ["Yes"],
    "PaymentMethod": ["Electronic check"],
    "MonthlyCharges": [80.0],
    "TotalCharges": [400.0]
})

prediction = pipeline.predict(new_customer)[0]
probability = pipeline.predict_proba(new_customer)[0, 1]

print("Prediction:", "Churn" if prediction == 1 else "Stay")
print(f"Probability of churn: {probability:.2%}")
print(f"Probability of staying: {1 - probability:.2%}")


## 21. Probability-based prediction

In [ ]:
probabilities = pipeline.predict_proba(new_customer)[0]

print(f"Probability of staying: {probabilities[0]:.2%}")
print(f"Probability of churning: {probabilities[1]:.2%}")


## 22. Business decision threshold

The model produces a probability. A business does not have to use the default 0.50 threshold.

For example, if the company wants to proactively contact more at-risk customers, it could use a lower threshold.


In [ ]:
threshold = 0.40

custom_prediction = (y_proba >= threshold).astype(int)

print("Threshold:", threshold)
print("Predicted churn cases:", custom_prediction.sum())


## 23. Quick model recap

In [ ]:
print("ML WORKFLOW")
print("1. Define the business problem")
print("2. Select meaningful features")
print("3. Separate features and target")
print("4. Split into training and testing data")
print("5. Preprocess numerical and categorical data")
print("6. Train Logistic Regression")
print("7. Evaluate with suitable metrics")
print("8. Predict for new customers")
print("9. Convert probabilities into business decisions")


# Challenge for you 🚀

Try changing the business problem while keeping the same workflow.

Examples:
- Student data → dropout prediction
- Transaction data → fraud prediction
- Loan data → default prediction

The algorithm may change, but the **Machine Learning workflow remains remarkably similar**.
